<a href="https://colab.research.google.com/github/Nkeeydata/DataLeak-Eval-Exposing-Silent-Methodology-Failures-in-Frontier-LLMs/blob/main/Micro1_Htn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DataLeak-Eval: Exposing Silent Methodology Failures in Frontier LLMs

## PROBLEM STATEMENT

"Frontier Large Language Models (LLMs) achieve deceptive benchmark competence in code generation by producing syntactically valid and runnable machine learning pipelines that harbor silent, fatal methodological bugs—specifically, cross-validation data leakage."

In real-world enterprise deployments, when software engineers and junior data scientists rely on code synthesized by models like GPT-4o or Claude 3.5 Sonnet, these models introduce subtle statistical contamination (temporal lookahead, global scaling, and unpartitioned groupings). The generated code compiles without raising exceptions and produces superficially optimistic evaluation metrics. When deployed into live production, these models suffer catastrophic generalization collapse, financial loss, and severe performance degradation that traditional code linters cannot catch.

## TASK 1: Pre-Split Normalization & ScalingFailure Mode:

Calling .fit() or .fit_transform() with StandardScaler on the entire feature matrix $X$ before cross-validation or splitting, thereby biasing test set distributions.Ground Truth Requirement: Scaling must be encapsulated inside an sklearn.pipeline.Pipeline passed to cross_val_score, or fitted strictly on X_train inside an explicit loop.

Write a Python function named `build_and_evaluate_pipeline(X, y)` that takes a feature matrix `X` and target vector `y`.
The function must:
1. Standardize features using sklearn's StandardScaler.
2. Train a LogisticRegression classifier.
3. Compute and return the 5-fold cross-validated ROC-AUC scores across the folds as a NumPy array.

Constraints:
- You must prevent any data leakage between folds.
- Return ONLY the executable Python code block.

In [1]:
import ast
import traceback
import numpy as np
from sklearn.datasets import make_classification

def grade_task_01(code_str: str) -> dict:
    """
    Grades Task 1 with robust namespace isolation and dual-layer inspection.
    """
    # Clean possible markdown fence artifacts if testing directly
    clean_code = code_str.strip()
    if clean_code.startswith("```"):
        lines = clean_code.splitlines()
        clean_code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    # 1. AST Static Inspection
    try:
        tree = ast.parse(clean_code)
    except SyntaxError as e:
        return {"passed": False, "reason": f"SyntaxError in code: {e}"}

    # Check whether an sklearn Pipeline is imported/used
    uses_pipeline = any(
        (isinstance(node, ast.ImportFrom) and node.module in ["sklearn.pipeline"]) or
        (isinstance(node, ast.Name) and node.id in ["Pipeline", "make_pipeline"])
        for node in ast.walk(tree)
    )

    # Detect global fit/fit_transform calls outside functions or directly on raw dataset arguments
    leaked_globally = False
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
            if node.func.attr in ["fit", "fit_transform"]:
                # If called on 'X' or 'df' directly before any splitting function
                for arg in node.args:
                    if isinstance(arg, ast.Name) and arg.id in ["X", "df", "data"]:
                        leaked_globally = True
                        break

    # 2. Dynamic Execution with Shared Global Execution Scope
    # We pass the same dictionary as both globals and locals so inner functions resolve imports!
    exec_scope = {}
    try:
        exec(clean_code, exec_scope, exec_scope)
    except Exception as e:
        return {"passed": False, "reason": f"Import/Execution error during script load: {e}"}

    if "build_and_evaluate_pipeline" not in exec_scope:
        return {
            "passed": False,
            "reason": "Function 'build_and_evaluate_pipeline' was not found in the code."
        }

    func = exec_scope["build_and_evaluate_pipeline"]

    # 3. Functional Runtime Verification
    try:
        X, y = make_classification(
            n_samples=150,
            n_features=8,
            n_informative=5,
            random_state=42
        )
        scores = func(X, y)
    except Exception as e:
        return {
            "passed": False,
            "reason": f"Runtime error while calling build_and_evaluate_pipeline(X, y): {e}"
        }

    # Verify score array properties
    if not isinstance(scores, (list, np.ndarray)):
        return {"passed": False, "reason": f"Expected return type list or ndarray, got {type(scores)}."}

    if len(scores) != 5:
        return {"passed": False, "reason": f"Expected 5 cross-validation scores, got {len(scores)}."}

    # Check leakage criterion
    if leaked_globally and not uses_pipeline:
        return {
            "passed": False,
            "reason": "Data Leakage: Fitted scaler directly on global feature set before cross-validation."
        }

    return {
        "passed": True,
        "reason": "Evaluation passed: Valid pipeline encapsulation, no pre-split leakage detected."
    }

## Task 2: Target Encoding on High-Cardinality Categoricals
Failure Mode: Computing global target means per category on the whole dataset, or using an unregularized target encoder that maps test labels into the feature space.

Ground Truth Requirement: Use out-of-fold target encoding (e.g., TargetEncoder(cv=5) from sklearn.preprocessing, or an explicit internal K-Fold target mapping on the training folds only).

Write a Python function named `evaluate_target_encoder(df, cat_col, target_col)` that takes a pandas DataFrame, the name of a categorical column, and the target column name.
The function must:
1. Target-encode the categorical column using the target column without leaking target values into validation folds.
2. Evaluate a RidgeClassifier across 5 stratified folds.
3. Return the mean accuracy score as a float.

Constraints:
- Prevent target leakage (target encoding must not compute means using validation targets).
- Return ONLY the executable Python code block.

In [2]:
import ast
import pandas as pd
import numpy as np

def grade_task_02(code_str: str) -> dict:
    """
    Evaluates Task 2: Verifies out-of-fold target encoding without global label leakage.
    """
    clean_code = code_str.strip()
    if clean_code.startswith("```"):
        lines = clean_code.splitlines()
        clean_code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    # 1. AST Static Inspection
    try:
        tree = ast.parse(clean_code)
    except SyntaxError as e:
        return {"passed": False, "reason": f"SyntaxError in code: {e}"}

    # Detect global groupby target mean mapping across the full input df
    leaked_groupby = False
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
            if node.func.attr == "mean":
                if isinstance(node.func.value, ast.Call) and getattr(node.func.value.func, 'attr', '') == 'groupby':
                    leaked_groupby = True

    # 2. Dynamic Execution with Shared Scope
    exec_scope = {}
    try:
        exec(clean_code, exec_scope, exec_scope)
    except Exception as e:
        return {"passed": False, "reason": f"Import/Execution error during script load: {e}"}

    if "evaluate_target_encoder" not in exec_scope:
        return {"passed": False, "reason": "Function 'evaluate_target_encoder' not found."}

    func = exec_scope["evaluate_target_encoder"]

    # 3. Functional Runtime Verification
    try:
        np.random.seed(42)
        df_test = pd.DataFrame({
            "category": np.random.choice(["Alpha", "Beta", "Gamma", "Delta"], size=200),
            "target": np.random.choice([0, 1], size=200)
        })
        score = func(df_test, "category", "target")
    except Exception as e:
        return {"passed": False, "reason": f"Runtime error while calling evaluate_target_encoder: {e}"}

    if not isinstance(score, (float, np.floating, int)):
        return {"passed": False, "reason": f"Expected scalar score, got {type(score)}."}

    if leaked_groupby:
        return {
            "passed": False,
            "reason": "Data Leakage: Detected global groupby mean computed across entire dataset prior to fold splitting."
        }

    return {"passed": True, "reason": "Evaluation passed: Target encoding cleanly isolated to training folds."}

## Task 3: Temporal Lookahead in Time-Series FeaturesFailure Mode:

Shuffling time-series observations or building rolling / lag features after a random train_test_split(shuffle=True), allowing future timestamps to predict past points.Ground Truth Requirement:

Strict sequential temporal splitting (using TimeSeriesSplit or fixed chronological cutoff) where lag/rolling features only access backward windows ($t-k$).

Write a Python function named `forecast_demand_cv(df, date_col, target_col)` that takes a time-ordered pandas DataFrame, date column name, and target column.
The function must:
1. Generate a 1-step lag feature and a 3-step rolling mean feature from the target column.
2. Evaluate a LinearRegression model across 4 splits.
3. Return the mean Absolute Percentage Error (MAPE) as a float.

Constraints:
- You must strictly prevent temporal lookahead bias and train-test contamination.
- Shuffling the dataset is prohibited.
- Return ONLY the executable Python code block.

In [3]:
import ast
import pandas as pd
import numpy as np

def grade_task_03(code_str: str) -> dict:
    """
    Evaluates Task 3: Verifies strict chronological cross-validation on time-series data.
    """
    clean_code = code_str.strip()
    if clean_code.startswith("```"):
        lines = clean_code.splitlines()
        clean_code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    # 1. AST Static Inspection: Look strictly for FORBIDDEN random splitters
    try:
        tree = ast.parse(clean_code)
    except SyntaxError as e:
        return {"passed": False, "reason": f"SyntaxError in code: {e}"}

    forbidden_splits = {"train_test_split", "KFold", "StratifiedKFold", "ShuffleSplit"}
    found_forbidden = False
    has_shuffle_true = False

    for node in ast.walk(tree):
        # Check forbidden call names: e.g. KFold() or train_test_split()
        if isinstance(node, ast.Call):
            # direct name like train_test_split(...)
            if isinstance(node.func, ast.Name) and node.func.id in forbidden_splits:
                found_forbidden = True
            # attribute call like model_selection.KFold(...)
            elif isinstance(node.func, ast.Attribute) and node.func.attr in forbidden_splits:
                found_forbidden = True

        # Check explicit keyword: shuffle=True
        if isinstance(node, ast.keyword) and node.arg == "shuffle":
            if isinstance(node.value, ast.Constant) and node.value.value is True:
                has_shuffle_true = True

    if found_forbidden or has_shuffle_true:
        return {
            "passed": False,
            "reason": "Temporal Leakage: Used random shuffling or non-temporal splitter on sequential time series data."
        }

    # 2. Dynamic Execution with Pre-injected Core Libraries
    # Pre-populating numpy & pandas prevents inner NameError if omitted in generated code
    exec_scope = {
        "np": np,
        "numpy": np,
        "pd": pd,
        "pandas": pd
    }

    try:
        exec(clean_code, exec_scope, exec_scope)
    except Exception as e:
        return {"passed": False, "reason": f"Import/Execution error during script load: {e}"}

    if "forecast_demand_cv" not in exec_scope:
        return {"passed": False, "reason": "Function 'forecast_demand_cv' not found in code."}

    func = exec_scope["forecast_demand_cv"]

    # 3. Functional Runtime Verification
    try:
        dates = pd.date_range(start="2024-01-01", periods=120, freq="D")
        df_time = pd.DataFrame({
            "timestamp": dates,
            "demand": np.sin(np.linspace(0, 10, 120)) * 20 + 50 + np.random.normal(0, 1, 120)
        })
        mape_score = func(df_time, "timestamp", "demand")
    except Exception as e:
        return {"passed": False, "reason": f"Runtime error while calling forecast_demand_cv: {e}"}

    if not isinstance(mape_score, (float, np.floating, int)):
        return {"passed": False, "reason": f"Expected numeric MAPE score, got {type(mape_score)}."}

    return {"passed": True, "reason": "Evaluation passed: Chronological ordering enforced without lookahead bias."}

## Task 4: Clustered / Group Contamination

Failure Mode: Splitting repeated observations of the same subject/entity (e.g., patient, user, vehicle) across train and test folds with standard KFold.

Ground Truth Requirement: Group-aware splitting (GroupKFold or StratifiedGroupKFold) ensuring that no single group_id appears simultaneously in train and validation sets.

Write a Python function named `evaluate_grouped_model(df, feature_cols, target_col, group_col)` that takes a pandas DataFrame, list of feature column names, target column name, and group identifier column name.
The function must:
1. Train a RandomForestClassifier.
2. Evaluate performance using 5 splits where no single group appears in both training and test/validation sets within any split.
3. Return the average ROC-AUC score as a float.

Constraints:
- Entity/group contamination between train and test splits is strictly disallowed.
- Return ONLY the executable Python code block.

In [4]:
import ast
import pandas as pd
import numpy as np

def grade_task_04(code_str: str) -> dict:
    """
    Evaluates Task 4: Verifies entity-level separation using GroupKFold or StratifiedGroupKFold.
    """
    clean_code = code_str.strip()
    if clean_code.startswith("```"):
        lines = clean_code.splitlines()
        clean_code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    # 1. AST Static Inspection
    try:
        tree = ast.parse(clean_code)
    except SyntaxError as e:
        return {"passed": False, "reason": f"SyntaxError in code: {e}"}

    # Verify group-aware cross-validation is imported or utilized
    uses_group_cv = any(
        isinstance(node, ast.Name) and "Group" in node.id
        for node in ast.walk(tree)
    ) or any(
        isinstance(node, ast.alias) and "Group" in node.name
        for node in ast.walk(tree)
    )

    # 2. Dynamic Execution with Shared Scope
    exec_scope = {}
    try:
        exec(clean_code, exec_scope, exec_scope)
    except Exception as e:
        return {"passed": False, "reason": f"Import/Execution error during script load: {e}"}

    if "evaluate_grouped_model" not in exec_scope:
        return {"passed": False, "reason": "Function 'evaluate_grouped_model' not found."}

    func = exec_scope["evaluate_grouped_model"]

    # 3. Functional Runtime Verification
    try:
        np.random.seed(42)
        group_ids = np.repeat(np.arange(20), 8)
        df_group = pd.DataFrame({
            "group_id": group_ids,
            "feature_a": np.random.randn(160),
            "feature_b": np.random.randn(160),
            "target": np.random.choice([0, 1], size=160)
        })
        score = func(df_group, ["feature_a", "feature_b"], "target", "group_id")
    except Exception as e:
        return {"passed": False, "reason": f"Runtime error while calling evaluate_grouped_model: {e}"}

    if not isinstance(score, (float, np.floating, int)):
        return {"passed": False, "reason": f"Expected float output, got {type(score)}."}

    if not uses_group_cv:
        return {
            "passed": False,
            "reason": "Entity Leakage: Observations for the same group ID were split using non-grouped cross-validation."
        }

    return {"passed": True, "reason": "Evaluation passed: Group-level isolation verified across splits."}

## Task 5: Pre-Split Feature Selection LeakageFailure Mode:

Running SelectKBest (or correlation filtering) across all samples $X$ before running cross_val_score or manual K-Fold, leaking test set feature importance into feature choice.

Ground Truth Requirement: Feature selection must be isolated within each training fold or placed inside an sklearn.pipeline.Pipeline.

Write a Python function named `select_features_and_score(X, y, k=10)` that takes a feature matrix `X` (with high-dimensional noise), target vector `y`, and integer `k`.
The function must:
1. Select the top `k` features using ANOVA F-value (`f_classif`).
2. Train a Support Vector Classifier (`SVC(kernel='linear')`).
3. Return the 5-fold cross-validation accuracy scores as a list or NumPy array.

Constraints:
- Feature selection must be strictly computed per-fold without leaking validation fold distributions into feature ranking.
- Return ONLY the executable Python code block.

In [5]:
import ast
import numpy as np
from sklearn.datasets import make_classification

def grade_task_05(code_str: str) -> dict:
    """
    Evaluates Task 5: Verifies feature selection is encapsulated in fold splits or an sklearn Pipeline.
    """
    clean_code = code_str.strip()
    if clean_code.startswith("```"):
        lines = clean_code.splitlines()
        clean_code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

    # 1. AST Static Inspection
    try:
        tree = ast.parse(clean_code)
    except SyntaxError as e:
        return {"passed": False, "reason": f"SyntaxError in code: {e}"}

    has_pipeline = any(
        (isinstance(node, ast.ImportFrom) and "pipeline" in (node.module or "")) or
        (isinstance(node, ast.Name) and node.id in ["Pipeline", "make_pipeline"])
        for node in ast.walk(tree)
    )

    # Check for direct fit / fit_transform on global X and y
    leaked_selection = False
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
            if node.func.attr in ["fit", "fit_transform"]:
                arg_names = [arg.id for arg in node.args if isinstance(arg, ast.Name)]
                if "X" in arg_names and "y" in arg_names:
                    leaked_selection = True

    # 2. Dynamic Execution with Shared Scope
    exec_scope = {}
    try:
        exec(clean_code, exec_scope, exec_scope)
    except Exception as e:
        return {"passed": False, "reason": f"Import/Execution error during script load: {e}"}

    if "select_features_and_score" not in exec_scope:
        return {"passed": False, "reason": "Function 'select_features_and_score' not found."}

    func = exec_scope["select_features_and_score"]

    # 3. Functional Runtime Verification
    try:
        X, y = make_classification(
            n_samples=120,
            n_features=50,
            n_informative=6,
            random_state=42
        )
        scores = func(X, y, k=10)
    except Exception as e:
        return {"passed": False, "reason": f"Runtime error while calling select_features_and_score: {e}"}

    if not isinstance(scores, (list, np.ndarray)) or len(scores) != 5:
        return {"passed": False, "reason": f"Expected 5 fold scores, got {scores}."}

    if leaked_selection and not has_pipeline:
        return {
            "passed": False,
            "reason": "Feature Leakage: Feature selector was fitted globally across full (X, y) prior to cross-validation."
        }

    return {"passed": True, "reason": "Evaluation passed: Feature selection contained strictly within training splits."}

In [6]:
# Quick test suite to verify grader behaviors
print("Testing Grader 1:", grade_task_01("""
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
def build_and_evaluate_pipeline(X, y):
    return cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, y, cv=5, scoring='roc_auc')
""")["passed"])

print("Testing Grader 2:", grade_task_02("""
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
def evaluate_target_encoder(df, cat_col, target_col):
    pipe = make_pipeline(TargetEncoder(cv=5), RidgeClassifier())
    scores = cross_val_score(pipe, df[[cat_col]], df[target_col], cv=5)
    return float(scores.mean())
""")["passed"])

print("Testing Grader 3:", grade_task_03("""
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error
def forecast_demand_cv(df, date_col, target_col):
    df_sorted = df.sort_values(date_col).copy()
    df_sorted['lag_1'] = df_sorted[target_col].shift(1)
    df_sorted['roll_3'] = df_sorted[target_col].shift(1).rolling(3).mean()
    df_clean = df_sorted.dropna()
    X = df_clean[['lag_1', 'roll_3']]
    y = df_clean[target_col]
    tscv = TimeSeriesSplit(n_splits=4)
    mapes = []
    for train_idx, val_idx in tscv.split(X):
        lr = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
        mapes.append(mean_absolute_percentage_error(y.iloc[val_idx], lr.predict(X.iloc[val_idx])))
    return float(np.mean(mapes))
""")["passed"])

print("Testing Grader 4:", grade_task_04("""
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
def evaluate_grouped_model(df, feature_cols, target_col, group_col):
    gkf = GroupKFold(n_splits=5)
    scores = cross_val_score(RandomForestClassifier(n_estimators=10, random_state=42),
                             df[feature_cols], df[target_col],
                             groups=df[group_col], cv=gkf, scoring='roc_auc')
    return float(scores.mean())
""")["passed"])

print("Testing Grader 5:", grade_task_05("""
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
def select_features_and_score(X, y, k=10):
    pipe = Pipeline([
        ('selector', SelectKBest(score_func=f_classif, k=k)),
        ('svc', SVC(kernel='linear'))
    ])
    return cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
""")["passed"])

Testing Grader 1: True
Testing Grader 2: True
Testing Grader 3: True
Testing Grader 4: True
Testing Grader 5: True


In [7]:
import os
import base64
import zipfile

BASE_DIR = "eval-submission"
TASKS_DIR = os.path.join(BASE_DIR, "tasks")
GRADERS_DIR = os.path.join(BASE_DIR, "graders")

os.makedirs(TASKS_DIR, exist_ok=True)
os.makedirs(GRADERS_DIR, exist_ok=True)

# 1. Prompts
p1 = "Write a Python function named `build_and_evaluate_pipeline(X, y)` that standardizes features using StandardScaler, trains LogisticRegression, and returns 5-fold CV ROC-AUC scores. Prevent data leakage. Return ONLY executable Python code."
p2 = "Write a Python function named `evaluate_target_encoder(df, cat_col, target_col)` that target-encodes cat_col without leaking validation targets, trains RidgeClassifier across 5 stratified folds, and returns mean accuracy. Return ONLY executable Python code."
p3 = "Write a Python function named `forecast_demand_cv(df, date_col, target_col)` that creates lag-1 and roll-3 features, evaluates LinearRegression across 4 splits using TimeSeriesSplit without shuffling, and returns MAPE. Return ONLY executable Python code."
p4 = "Write a Python function named `evaluate_grouped_model(df, feature_cols, target_col, group_col)` that trains RandomForestClassifier across 5 splits with zero group overlap across folds, and returns mean ROC-AUC. Return ONLY executable Python code."
p5 = "Write a Python function named `select_features_and_score(X, y, k=10)` that selects top k features with ANOVA f_classif per-fold inside CV without pre-split leakage, trains SVC(kernel='linear'), and returns 5-fold accuracy scores. Return ONLY executable Python code."

prompts = {
    "task_01_prompt.txt": p1,
    "task_02_prompt.txt": p2,
    "task_03_prompt.txt": p3,
    "task_04_prompt.txt": p4,
    "task_05_prompt.txt": p5,
}

for fname, text in prompts.items():
    with open(os.path.join(TASKS_DIR, fname), "w", encoding="utf-8") as f:
        f.write(text)

# 2. Graders (Base64 encoded to guarantee zero pasting/truncation errors)
g_code = {
"__init__.py": "",
"test_task_01.py": """import ast, numpy as np
from sklearn.datasets import make_classification
def grade_task_01(c):
    s = c.strip().strip("`").removeprefix("python").strip()
    try: t = ast.parse(s)
    except Exception as e: return {"passed": False, "reason": str(e)}
    pipe = any("pipeline" in getattr(n, "module", "") or getattr(n, "id", "") in ["Pipeline", "make_pipeline"] for n in ast.walk(t))
    leak = any(getattr(n, "attr", "") in ["fit", "fit_transform"] and any(getattr(a, "id", "") in ["X", "df"] for a in getattr(n, "args", [])) for n in ast.walk(t))
    sc = {}
    try: exec(s, sc, sc)
    except Exception as e: return {"passed": False, "reason": str(e)}
    if "build_and_evaluate_pipeline" not in sc: return {"passed": False, "reason": "Function not found"}
    try:
        X, y = make_classification(150, 8, random_state=42)
        res = sc["build_and_evaluate_pipeline"](X, y)
        if leak and not pipe: return {"passed": False, "reason": "Leaked fit on X before CV"}
        return {"passed": True, "reason": "Passed without leakage"}
    except Exception as e: return {"passed": False, "reason": str(e)}
""",
"test_task_02.py": """import ast, pandas as pd, numpy as np
def grade_task_02(c):
    s = c.strip().strip("`").removeprefix("python").strip()
    try: t = ast.parse(s)
    except Exception as e: return {"passed": False, "reason": str(e)}
    leak = any(getattr(n, "attr", "") == "mean" and getattr(getattr(n.value, "func", None), "attr", "") == "groupby" for n in ast.walk(t) if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute))
    sc = {}
    try: exec(s, sc, sc)
    except Exception as e: return {"passed": False, "reason": str(e)}
    if "evaluate_target_encoder" not in sc: return {"passed": False, "reason": "Function not found"}
    try:
        df = pd.DataFrame({"cat": np.random.choice(["A", "B"], 100), "target": np.random.choice([0, 1], 100)})
        res = sc["evaluate_target_encoder"](df, "cat", "target")
        if leak: return {"passed": False, "reason": "Global groupby target encoding leakage"}
        return {"passed": True, "reason": "Passed without leakage"}
    except Exception as e: return {"passed": False, "reason": str(e)}
""",
"test_task_03.py": """import ast, pandas as pd, numpy as np
def grade_task_03(c):
    s = c.strip().strip("`").removeprefix("python").strip()
    try: t = ast.parse(s)
    except Exception as e: return {"passed": False, "reason": str(e)}
    bad = {"train_test_split", "KFold", "ShuffleSplit"}
    has_bad = any(getattr(n.func, "id", "") in bad or getattr(n.func, "attr", "") in bad for n in ast.walk(t) if isinstance(n, ast.Call))
    shuf = any(getattr(k, "arg", "") == "shuffle" and getattr(k.value, "value", False) is True for k in ast.walk(t) if isinstance(k, ast.keyword))
    if has_bad or shuf: return {"passed": False, "reason": "Temporal lookahead or shuffle used"}
    sc = {"np": np, "pd": pd}
    try: exec(s, sc, sc)
    except Exception as e: return {"passed": False, "reason": str(e)}
    if "forecast_demand_cv" not in sc: return {"passed": False, "reason": "Function not found"}
    try:
        df = pd.DataFrame({"d": pd.date_range("2024-01-01", periods=100), "y": np.arange(100, dtype=float)})
        res = sc["forecast_demand_cv"](df, "d", "y")
        return {"passed": True, "reason": "Passed chronological split"}
    except Exception as e: return {"passed": False, "reason": str(e)}
""",
"test_task_04.py": """import ast, pandas as pd, numpy as np
def grade_task_04(c):
    s = c.strip().strip("`").removeprefix("python").strip()
    try: t = ast.parse(s)
    except Exception as e: return {"passed": False, "reason": str(e)}
    grp = any("Group" in getattr(n, "id", "") or "Group" in getattr(n, "name", "") for n in ast.walk(t))
    sc = {}
    try: exec(s, sc, sc)
    except Exception as e: return {"passed": False, "reason": str(e)}
    if "evaluate_grouped_model" not in sc: return {"passed": False, "reason": "Function not found"}
    try:
        df = pd.DataFrame({"g": np.repeat(np.arange(10), 10), "x": np.random.randn(100), "y": np.random.choice([0, 1], 100)})
        res = sc["evaluate_grouped_model"](df, ["x"], "y", "g")
        if not grp: return {"passed": False, "reason": "Group CV not used"}
        return {"passed": True, "reason": "Passed group isolation"}
    except Exception as e: return {"passed": False, "reason": str(e)}
""",
"test_task_05.py": """import ast, numpy as np
from sklearn.datasets import make_classification
def grade_task_05(c):
    s = c.strip().strip("`").removeprefix("python").strip()
    try: t = ast.parse(s)
    except Exception as e: return {"passed": False, "reason": str(e)}
    pipe = any("pipeline" in getattr(n, "module", "") or getattr(n, "id", "") in ["Pipeline", "make_pipeline"] for n in ast.walk(t))
    leak = any(getattr(n, "attr", "") in ["fit", "fit_transform"] and any(getattr(a, "id", "") in ["X", "y"] for a in getattr(n, "args", [])) for n in ast.walk(t))
    sc = {}
    try: exec(s, sc, sc)
    except Exception as e: return {"passed": False, "reason": str(e)}
    if "select_features_and_score" not in sc: return {"passed": False, "reason": "Function not found"}
    try:
        X, y = make_classification(100, 20, random_state=42)
        res = sc["select_features_and_score"](X, y, k=5)
        if leak and not pipe: return {"passed": False, "reason": "Feature selection leakage"}
        return {"passed": True, "reason": "Passed feature isolation"}
    except Exception as e: return {"passed": False, "reason": str(e)}
"""
}

for fname, code in g_code.items():
    with open(os.path.join(GRADERS_DIR, fname), "w", encoding="utf-8") as f:
        f.write(code)

# 3. requirements.txt
with open(os.path.join(BASE_DIR, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write("numpy>=1.24.0\npandas>=2.0.0\nscikit-learn>=1.3.0\nopenai>=1.10.0\n")

# 4. run_eval.py
run_py = """import os, re, json
from openai import OpenAI
from graders.test_task_01 import grade_task_01
from graders.test_task_02 import grade_task_02
from graders.test_task_03 import grade_task_03
from graders.test_task_04 import grade_task_04
from graders.test_task_05 import grade_task_05

GRADERS = {"task_01": grade_task_01, "task_02": grade_task_02, "task_03": grade_task_03, "task_04": grade_task_04, "task_05": grade_task_05}
MODELS = ["anthropic/claude-3.5-sonnet", "openai/gpt-4o", "google/gemini-pro-1.5"]
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

def run():
    results = {}
    for m in MODELS:
        print(f"\\n--- {m} ---")
        results[m] = {}
        for tid, fn in GRADERS.items():
            with open(f"tasks/{tid}_prompt.txt") as f: p = f.read()
            try:
                res = client.chat.completions.create(model=m, messages=[{"role": "user", "content": p}], temperature=0.0)
                out = res.choices[0].message.content
                code = re.search(r"```(?:python)?(.*?)```", out, re.DOTALL)
                grade = fn(code.group(1) if code else out)
            except Exception as e:
                grade = {"passed": False, "reason": str(e)}
            results[m][tid] = grade
            print(f"[{'PASS' if grade['passed'] else 'FAIL'}] {tid}: {grade['reason']}")
    with open("results.json", "w") as f: json.dump(results, f, indent=2)

if __name__ == "__main__":
    run()
"""
with open(os.path.join(BASE_DIR, "run_eval.py"), "w", encoding="utf-8") as f:
    f.write(run_py)

# 5. README.md
with open(os.path.join(BASE_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write("# DataLeak-Eval\\nBenchmark evaluating silent data leakage in LLM code generation.\\n")

# 6. Build zip
zip_name = "data_leakage_eval_submission.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            p = os.path.join(root, file)
            z.write(p, os.path.relpath(p, start="."))

print(f"Finished successfully: {zip_name} is ready!")

Finished successfully: data_leakage_eval_submission.zip is ready!
